# Logistic Regression - Model Training

Entrenamiento de modelos de Regresión Logística (BALANCED vs RAW) con GridSearchCV.

> **¿Por qué Regresión Logística?**  
> Es un modelo lineal que estima la *probabilidad* de pertenecer a cada clase. A diferencia de XGBoost o RF, no construye árboles — aprende una frontera de decisión lineal en el espacio de features. Es interpretable, rápido y sirve como buen *baseline* para comparar contra modelos más complejos.

## 1. Imports

In [1]:
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    precision_recall_fscore_support
)
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported")

✓ Libraries imported


## 2. Configure Paths

In [2]:
BASE_PATH = Path('../../..').resolve()
DATA_PATH = BASE_PATH / 'data' / 'processed' / 'pickle'
MODELS_PATH = BASE_PATH / 'models' / 'logistic_regression'
RESULTS_PATH = BASE_PATH / 'data' / 'results' / 'logistic_regression'

MODELS_PATH.mkdir(parents=True, exist_ok=True)
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

print(f"✓ Paths configured")
print(f"  Models  → {MODELS_PATH}")
print(f"  Results → {RESULTS_PATH}")

✓ Paths configured
  Models  → /home/pablo/Desktop/Estressss/pdg/models/logistic_regression
  Results → /home/pablo/Desktop/Estressss/pdg/data/results/logistic_regression


## 3. Load Data

In [3]:
with open(DATA_PATH / 'X_train_balanced.pkl', 'rb') as f:
    X_train_balanced = pickle.load(f)
with open(DATA_PATH / 'y_train_balanced.pkl', 'rb') as f:
    y_train_balanced = pickle.load(f)

with open(DATA_PATH / 'X_train_raw.pkl', 'rb') as f:
    X_train_raw = pickle.load(f)
with open(DATA_PATH / 'y_train_raw.pkl', 'rb') as f:
    y_train_raw = pickle.load(f)

if isinstance(X_train_balanced, pd.DataFrame): X_train_balanced = X_train_balanced.values
if isinstance(X_train_raw,   pd.DataFrame): X_train_raw   = X_train_raw.values
if isinstance(y_train_balanced, pd.DataFrame): y_train_balanced = y_train_balanced.iloc[:, 0]
if isinstance(y_train_raw,   pd.DataFrame): y_train_raw   = y_train_raw.iloc[:, 0]


print(f"✓ BALANCED Training set : {X_train_balanced.shape}")
print(f"✓ RAW   Training set : {X_train_raw.shape}")
print(f"Class distribution:")
print(f"  BALANCED Train : {dict(pd.Series(y_train_balanced).value_counts().sort_index())}")
print(f"  RAW   Train : {dict(pd.Series(y_train_raw).value_counts().sort_index())}")

✓ BALANCED Training set : (898, 651)
✓ RAW   Training set : (699, 651)
Class distribution:
  BALANCED Train : {0: np.int64(633), 1: np.int64(265)}
  RAW   Train : {0: np.int64(633), 1: np.int64(66)}


## 4. Hyperparameter Tuning - Grid Search

> **Parámetros clave de Logistic Regression:**
>
> - **C**: controla la regularización. C pequeño = más regularización (modelo más simple, menos overfitting). C grande = menos regularización (modelo más flexible).
> - **penalty**: tipo de regularización.  penaliza coeficientes grandes.  puede llevar coeficientes a cero (selección automática de features).
> - **solver**: algoritmo de optimización.  soporta tanto  como  y es eficiente con datasets grandes.
>
> ⚡ **Este Grid Search es mucho más rápido que RF o XGBoost** — solo 4×2 = 8 combinaciones × 10 folds = 80 entrenamientos.

In [4]:
print("=" * 70)
print("HYPERPARAMETER TUNING - GRID SEARCH")
print("=" * 70)
print("Configuration:")
print("  • Metric: Recall Macro (10-fold Stratified CV)")
print("  • Cross-validation: StratifiedKFold (n_splits=10)")
print("  • Strategy: Tune C (regularization) and penalty type")

cv_stratified = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Grid pequeño y enfocado — LR no necesita tantos hiperparámetros como RF o XGBoost
param_grid = {
    'C':       [0.01, 0.1, 1.0, 10.0],   # fuerza de regularización inversa
    'penalty': ['l1', 'l2'],              # tipo de regularización
}

print(f"Parameter grid:")
for k, v in param_grid.items():
    print(f"  {k}: {v}")

HYPERPARAMETER TUNING - GRID SEARCH
Configuration:
  • Metric: Recall Macro (10-fold Stratified CV)
  • Cross-validation: StratifiedKFold (n_splits=10)
  • Strategy: Tune C (regularization) and penalty type
Parameter grid:
  C: [0.01, 0.1, 1.0, 10.0]
  penalty: ['l1', 'l2']


## 5. Base Model and Grid Search Function

In [5]:
def create_base_model():
    """Base Logistic Regression model.
    
    - multi_class='multinomial': estrategia nativa para 3+ clases
    - class_weight='balanced':   compensa el desbalance de clases
    - solver='saga':             soporta l1 y l2, eficiente con datos grandes
    - max_iter=1000:             suficientes iteraciones para que converja
    """
    return LogisticRegression(
        class_weight='balanced',
        solver='saga',
        max_iter=1000,
        random_state=42,
    )


def run_grid_search(X_train, y_train, model_name):
    """Execute GridSearchCV for BALANCED or RAW model."""
    print(f"{'-'*70}")
    print(f"Grid Search: {model_name} Model")
    print(f"{'-'*70}")

    sample_weights = compute_sample_weight('balanced', y_train)

    grid_search = GridSearchCV(
        estimator=create_base_model(),
        param_grid=param_grid,
        cv=cv_stratified,
        scoring='recall_macro',
        n_jobs=-1,
        verbose=1,
    )

    print(f"Searching optimal hyperparameters...")
    grid_search.fit(X_train, y_train, sample_weight=sample_weights)

    print(f"✓ Best hyperparameters ({model_name}):")
    for param, value in grid_search.best_params_.items():
        print(f"    {param}: {value}")
    print(f"✓ Best Recall Macro (CV): {grid_search.best_score_:.4f}")

    cv_results = pd.DataFrame(grid_search.cv_results_)
    suffix = 'balanced' if 'BALANCED' in model_name else 'raw'
    cv_results.to_csv(RESULTS_PATH / f'gridsearch_results_{suffix}.csv', index=False)
    print(f"✓ Grid Search results saved")

    return grid_search.best_estimator_, grid_search

## 6. Run Grid Search for Both Models

In [6]:
model_balanced, gs_balanced = run_grid_search(X_train_balanced, y_train_balanced, 'BALANCED')
model_raw,   gs_raw   = run_grid_search(X_train_raw,   y_train_raw,   'RAW')

print(f"" + "="*70)
print(f"✓ Both models trained successfully")
print(f"="*70)

----------------------------------------------------------------------
Grid Search: BALANCED Model
----------------------------------------------------------------------
Searching optimal hyperparameters...
Fitting 10 folds for each of 8 candidates, totalling 80 fits
✓ Best hyperparameters (BALANCED):
    C: 0.1
    penalty: l2
✓ Best Recall Macro (CV): 0.9420
✓ Grid Search results saved
----------------------------------------------------------------------
Grid Search: RAW Model
----------------------------------------------------------------------
Searching optimal hyperparameters...
Fitting 10 folds for each of 8 candidates, totalling 80 fits
✓ Best hyperparameters (RAW):
    C: 0.1
    penalty: l1
✓ Best Recall Macro (CV): 0.6666
✓ Grid Search results saved
✓ Both models trained successfully


## 7. Training Set Performance

> ⚠️ A diferencia de RF, Logistic Regression **no memoriza** los datos de entrenamiento — es un modelo lineal. Si ves métricas moderadas aquí (ej. 0.65–0.75), es normal y saludable. Indica que el modelo aprendió patrones reales, no ruido.

In [7]:
print("" + "="*70)
print("TRAINING SET PERFORMANCE")
print("="*70)

y_train_pred_balanced = model_balanced.predict(X_train_balanced)
print(f"BALANCED Model:")
print(f"  Accuracy : {accuracy_score(y_train_balanced, y_train_pred_balanced):.4f}")
print(f"  Precision: {precision_score(y_train_balanced, y_train_pred_balanced, average='macro', zero_division=0):.4f}")
print(f"  Recall   : {recall_score(y_train_balanced, y_train_pred_balanced, average='macro', zero_division=0):.4f}")
print(f"  F1-Score : {f1_score(y_train_balanced, y_train_pred_balanced, average='macro', zero_division=0):.4f}")

y_train_pred_raw = model_raw.predict(X_train_raw)
print(f"RAW Model:")
print(f"  Accuracy : {accuracy_score(y_train_raw, y_train_pred_raw):.4f}")
print(f"  Precision: {precision_score(y_train_raw, y_train_pred_raw, average='macro', zero_division=0):.4f}")
print(f"  Recall   : {recall_score(y_train_raw, y_train_pred_raw, average='macro', zero_division=0):.4f}")
print(f"  F1-Score : {f1_score(y_train_raw, y_train_pred_raw, average='macro', zero_division=0):.4f}")

TRAINING SET PERFORMANCE
BALANCED Model:
  Accuracy : 0.9788
  Precision: 0.9673
  Recall   : 0.9839
  F1-Score : 0.9750
RAW Model:
  Accuracy : 0.8956
  Precision: 0.7342
  Recall   : 0.9220
  F1-Score : 0.7861


## 8. Per-Class Metrics (Training Set)

In [8]:
print("" + "="*70)
print("PER-CLASS METRICS (TRAINING SET)")
print("="*70)

prec_s, rec_s, f1_s, sup_s = precision_recall_fscore_support(y_train_balanced, y_train_pred_balanced, average=None)
print(f"BALANCED Model:")
print(pd.DataFrame({'Class': [0,1], 'Precision': prec_s, 'Recall': rec_s, 'F1-Score': f1_s, 'Support': sup_s}).to_string(index=False))

prec_r, rec_r, f1_r, sup_r = precision_recall_fscore_support(y_train_raw, y_train_pred_raw, average=None)
print(f"RAW Model:")
print(pd.DataFrame({'Class': [0,1], 'Precision': prec_r, 'Recall': rec_r, 'F1-Score': f1_r, 'Support': sup_r}).to_string(index=False))

PER-CLASS METRICS (TRAINING SET)
BALANCED Model:
 Class  Precision   Recall  F1-Score  Support
     0   0.998377 0.971564  0.984788      633
     1   0.936170 0.996226  0.965265      265
RAW Model:
 Class  Precision   Recall  F1-Score  Support
     0   0.994700 0.889415  0.939116      633
     1   0.473684 0.954545  0.633166       66


## 9. Save Models

In [9]:
with open(MODELS_PATH / 'modelo_logistic_regression_balanced.pkl', 'wb') as f:
    pickle.dump(model_balanced, f)
print(f"✓ BALANCED model saved")

with open(MODELS_PATH / 'modelo_logistic_regression_raw.pkl', 'wb') as f:
    pickle.dump(model_raw, f)
print(f"✓ RAW model saved")

print(f"✅ Both models ready for evaluation")
print(f"   → Run logistic_regression_evaluation.ipynb next")

✓ BALANCED model saved
✓ RAW model saved
✅ Both models ready for evaluation
   → Run logistic_regression_evaluation.ipynb next
